In [2]:
# Imports
import numpy as np
from scipy.sparse import csr_matrix, save_npz
import pandas as pd
from pathlib import Path

## Part 1: Initial Processing

### Prerequisite: Data Fetching

This relies on the [Last.fm Dataset-360k users](http://ocelma.net/MusicRecommendationDataset/lastfm-360K.html).
The dataset contains `<user, artist-mbid, artist-name, total-plays>` tuples for ~360k users, collected in Fall 2008 by Òscar Celma with permission from Last.fm.
`artist-mbid` is a unique identifier for music artists in the crowdsourced music database [Musicbrainz](https://musicbrainz.org). There are ~187k artists with MBIDs and ~107k artists without MBIDs in the dataset.

The data can be downloaded at [http://mtg.upf.edu/static/datasets/last.fm/lastfm-dataset-360K.tar.gz](http://mtg.upf.edu/static/datasets/last.fm/lastfm-dataset-360K.tar.gz).
It is about ~1.5 GB and not included in this repository.

Read data into dataframe

In [3]:
filename = Path("raw_data") / "lastfm_360k_dataset.tsv"
if filename.exists():
    fm_df = pd.read_csv(filename, sep='\t')
else:
    raise FileNotFoundError("You must download the data to raw_data/lastfm_360k_dataset.tsv from the link http://mtg.upf.edu/static/datasets/last.fm/lastfm-dataset-360K.tar.gz")


In [4]:
fm_df.head(5)

,00000c289a1829a808ac09c00daf10bc3c4e223b,3bd73256-3905-4f3a-97e2-8b341527f805,betty blowtorch,2137
0,00000c289a1829a808ac09c00daf10bc3c4e223b,f2fb0ff0-5679-42ec-a55c-15109ce6e320,die Ärzte,1099
1,00000c289a1829a808ac09c00daf10bc3c4e223b,b3ae82c2-e60b-4551-a76d-6620f1b456aa,melissa etheridge,897
2,00000c289a1829a808ac09c00daf10bc3c4e223b,3d6bbeb7-f90e-4d10-b440-e153c0d10b53,elvenking,717
3,00000c289a1829a808ac09c00daf10bc3c4e223b,bbd2ffd7-17f4-4506-8572-c1ea58c3f9a8,juliette & the licks,706
4,00000c289a1829a808ac09c00daf10bc3c4e223b,8bfac288-ccc5-448d-9573-c33ea2aa5c30,red hot chili peppers,691


Name columns

In [5]:
fm_df.columns = ["user_sha1", "artist_mbid", "artist", "play_count"]
fm_df.head(5)

,user_sha1,artist_mbid,artist,play_count
0,00000c289a1829a808ac09c00daf10bc3c4e223b,f2fb0ff0-5679-42ec-a55c-15109ce6e320,die Ärzte,1099
1,00000c289a1829a808ac09c00daf10bc3c4e223b,b3ae82c2-e60b-4551-a76d-6620f1b456aa,melissa etheridge,897
2,00000c289a1829a808ac09c00daf10bc3c4e223b,3d6bbeb7-f90e-4d10-b440-e153c0d10b53,elvenking,717
3,00000c289a1829a808ac09c00daf10bc3c4e223b,bbd2ffd7-17f4-4506-8572-c1ea58c3f9a8,juliette & the licks,706
4,00000c289a1829a808ac09c00daf10bc3c4e223b,8bfac288-ccc5-448d-9573-c33ea2aa5c30,red hot chili peppers,691


Create a pd.Series of unique users

In [6]:
all_users = fm_df["user_sha1"].drop_duplicates()
all_users.head(5)


0      00000c289a1829a808ac09c00daf10bc3c4e223b
48     00001411dc427966b17297bf4d69e7e193135d89
99     00004d2ac9316e22dc007ab2243d6fcb239e707d
145    000063d3fe1cf2ba248b9e3c3f0334845a27a6bf
194    00007a47085b9aab8af55f52ec8846ac479ac4fe
Name: user_sha1, dtype: str

In [7]:
all_users.size

358868

Create a random ordered index for the users series

In [8]:
rng = np.random.default_rng(1235)

order = rng.permutation(len(all_users))

all_users.index = order
all_users.head(5)

320894    00000c289a1829a808ac09c00daf10bc3c4e223b
197937    00001411dc427966b17297bf4d69e7e193135d89
136568    00004d2ac9316e22dc007ab2243d6fcb239e707d
266843    000063d3fe1cf2ba248b9e3c3f0334845a27a6bf
141122    00007a47085b9aab8af55f52ec8846ac479ac4fe
Name: user_sha1, dtype: str

Sort the users by their random indices

In [9]:
sorted_users = all_users.sort_index()
sorted_users.head(5)

0    aa7032a8a477cfca85a56046f51727e776548695
1    004a6f5cedf5d2cfd8c8e781fb1a91aa02b53533
2    c1c593b8aaa4e0c4f1d3a2345ed2ed8b01ffe38e
3    5b18a9c516ca94aeae86139bb3ff885e900da37d
4    0087182c5fbeaaa49206d3486d30530410a15f85
Name: user_sha1, dtype: str

Take just the first $n$ entries in the randomly-ordered list of all users

In [10]:
selected_users = sorted_users[0:20000]

Create a new dataframe with the subset of the data, containing only the specified $n$ users

In [11]:
fm_df_subset = fm_df[fm_df["user_sha1"].isin(selected_users)]
fm_df_subset.head(5)

,user_sha1,artist_mbid,artist,play_count
704,000215d3060a5b0ab7b3c415d49ec579100d4c87,12327d75-47d5-45d9-84c2-3760b9210c17,syd barrett,176
705,000215d3060a5b0ab7b3c415d49ec579100d4c87,65041e06-83d2-4987-ae52-c17a915fc82a,spiritualized,157
706,000215d3060a5b0ab7b3c415d49ec579100d4c87,1b5189d8-28fd-4c97-a04a-37c49b1ccfa6,the boo radleys,97
707,000215d3060a5b0ab7b3c415d49ec579100d4c87,381e8434-109a-439c-8acc-5d328359f2e0,the brian jonestown massacre,93
708,000215d3060a5b0ab7b3c415d49ec579100d4c87,a74b1b7f-71a5-4011-9441-d0b5e4122711,radiohead,81


In [12]:
# Inspect the DataFrame
print(fm_df_subset.shape)
print(fm_df_subset.columns.tolist())
print(fm_df_subset.dtypes)

(977290, 4)
['user_sha1', 'artist_mbid', 'artist', 'play_count']
user_sha1        str
artist_mbid      str
artist           str
play_count     int64
dtype: object


In [13]:
# Set column types
fm_df_subset['user_sha1'] = fm_df_subset['user_sha1'].astype('category')
fm_df_subset['artist_mbid'] = fm_df_subset['artist_mbid'].astype('category')

We only need `user_sha1`, `artist_mbid`, and `play_count` for the sparse matrix. However, so we can refer to the actual artists in the future, we can preserve a mapping of artist mbid to artist name.

In [14]:
mbid_artist_map = dict(zip(fm_df_subset['artist_mbid'], fm_df_subset['artist']))
fm_df_subset = fm_df_subset.drop(columns=['artist'])
fm_df_subset['user_sha1'] = fm_df_subset['user_sha1'].cat.remove_unused_categories()

In [15]:
# Check column names and dtypes
print(fm_df_subset.columns.tolist())
print(fm_df_subset.dtypes)

['user_sha1', 'artist_mbid', 'play_count']
user_sha1      category
artist_mbid    category
play_count        int64
dtype: object


In [16]:
# count rows with the same user-artist pairing
# these may occur because the raw data separates based on artist strings, but several artist strings may
# be matched to the same artist mbid in the MusicBrainz database
fm_df_subset.duplicated(['user_sha1', 'artist_mbid']).sum()

np.int64(17)

## Part 2: Grouping

In [17]:
# create group for each user-artist_mbid pair
grouped_df = fm_df_subset.groupby(['user_sha1', 'artist_mbid'], observed=True)
# show total playcounts for each pair
totals = grouped_df['play_count'].sum().reset_index()
totals.head(5)

,user_sha1,artist_mbid,play_count
0,000215d3060a5b0ab7b3c415d49ec579100d4c87,0428e118-88aa-47a7-8776-49d1a5c34e77,15
1,000215d3060a5b0ab7b3c415d49ec579100d4c87,04da193a-a956-4940-a6ca-e6021c7861c2,15
2,000215d3060a5b0ab7b3c415d49ec579100d4c87,12327d75-47d5-45d9-84c2-3760b9210c17,176
3,000215d3060a5b0ab7b3c415d49ec579100d4c87,1b5189d8-28fd-4c97-a04a-37c49b1ccfa6,97
4,000215d3060a5b0ab7b3c415d49ec579100d4c87,1cc5adcd-1422-4b5c-a3cd-3ecd4f43f506,9


Verify there are no duplicates left

In [18]:
totals.duplicated(['user_sha1', 'artist_mbid']).sum()

np.int64(0)

# Part 3: Saving as sparse matrix

In [25]:
# Convert to categoricals to get integer codes for each user/artist
user_cat = totals['user_sha1'].astype('category')
artist_cat = totals['artist_mbid'].astype('category')

user_idx = user_cat.cat.codes
artist_idx = artist_cat.cat.codes

# Build the sparse matrix: rows = users, columns = artists
sparse_matrix = csr_matrix(
    (totals['play_count'], (user_idx, artist_idx)),
    shape=(user_cat.cat.categories.size, artist_cat.cat.categories.size)
)

# Keep the labels so you can map back
users = user_cat.cat.categories
artists = artist_cat.cat.categories
artist_labels = pd.Index([mbid_artist_map[artist] for artist in artists])

print(sparse_matrix.shape)
print(f"Non-zero entries: {sparse_matrix.nnz}")

(20000, 66231)
Non-zero entries: 964617


In [43]:
artists.isna().any()

np.False_

In [35]:
u = users.get_loc('000215d3060a5b0ab7b3c415d49ec579100d4c87')
a = artists.get_loc('1b5189d8-28fd-4c97-a04a-37c49b1ccfa6')
# alternative: a = artist_labels.get_loc('the boo radleys')
sparse_matrix[u, a] # Should return 97

np.int64(97)

In [30]:
artist_labels[7040]

'the boo radleys'

# Part 4: Saving Sparse Matrix to Disk

In [36]:
save_npz('data/tidied_360k/user_artist_matrix.npz', sparse_matrix)

# Save the labels too, so you can map rows/columns back to names
np.save('data/tidied_360k/users.npy', users.to_numpy(), allow_pickle=True)
np.save('data/tidied_360k/artists.npy', artists.to_numpy(), allow_pickle=True)
np.save('data/tidied_360k/artist_labels.npy', artist_labels.to_numpy(), allow_pickle=True)